# Notebook 09: ML Platform Design

## Why This Matters

ML platforms are the infrastructure layer that makes data scientists and ML engineers 10× more productive. Uber built Michelangelo, Meta built FBLearner Flow, Airbnb built Bighead, and LinkedIn built Pro-ML—all internal ML platforms that standardized how models are trained, served, and monitored across hundreds of use cases. As companies scale their ML teams, the cost of each team reinventing the training-serving-monitoring wheel becomes unsustainable. Understanding ML platform design—what functionality to centralize, what to leave to teams, and how to balance standardization with flexibility—is a senior ML engineering concern. This topic appears frequently in staff/principal-level system design interviews.

## 1. What Does an ML Platform Provide?

An ML platform is the shared infrastructure layer between raw data and production ML models. It standardizes the common workflows while providing enough flexibility for diverse ML use cases. The key design tension is: how much to standardize vs how much to allow customization.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(16, 11))
ax.set_xlim(0, 16); ax.set_ylim(0, 11); ax.axis('off')
ax.set_title('ML Platform Architecture: The Core Components', fontsize=14, fontweight='bold')

def box(ax, x, y, w, h, title, body, color, fontsize=8.5):
    r = FancyBboxPatch((x,y), w, h, boxstyle="round,pad=0.12",
                       facecolor=color, edgecolor='white', linewidth=2, alpha=0.9)
    ax.add_patch(r)
    ax.text(x+w/2, y+h-0.35, title, ha='center', va='center',
            fontsize=fontsize, fontweight='bold', color='white')
    lines = body.split('\n')
    for i, line in enumerate(lines):
        ax.text(x+0.25, y+h-0.75 - i*0.38, line, fontsize=7.5, color='white', alpha=0.92)

# Data Layer
box(ax, 0.3, 8.8, 15.4, 1.8, 'DATA LAYER',
    '• Data Lake (S3/GCS)  • Data Warehouse (Snowflake/BQ)  • Stream Processing (Kafka/Flink)  • Feature Store (Feast/Tecton)',
    '#2C3E50')

# Feature Engineering
box(ax, 0.3, 6.8, 4.5, 1.8, 'FEATURE ENGINEERING',
    '• Feature pipelines\n• Transform library\n• Feature versioning\n• PIT join utilities',
    '#2980B9')

# Training Platform
box(ax, 5.2, 6.8, 4.5, 1.8, 'TRAINING PLATFORM',
    '• Job scheduler\n• Distributed training\n• HP tuning service\n• Compute resource mgmt',
    '#8E44AD')

# Experiment Tracking
box(ax, 10.1, 6.8, 5.6, 1.8, 'EXPERIMENT TRACKING',
    '• Run management (MLflow)\n• Metric logging\n• Model versioning\n• Lineage tracking',
    '#16A085')

# Model Registry
box(ax, 0.3, 4.8, 4.5, 1.8, 'MODEL REGISTRY',
    '• Model artifacts storage\n• Version management\n• Stage (staging/prod)\n• Access control',
    '#E67E22')

# Serving Platform
box(ax, 5.2, 4.8, 4.5, 1.8, 'SERVING PLATFORM',
    '• Online model server\n• Batch scoring\n• A/B test routing\n• Auto-scaling',
    '#E74C3C')

# Monitoring
box(ax, 10.1, 4.8, 5.6, 1.8, 'MONITORING',
    '• Feature drift detection\n• Prediction monitoring\n• Business metrics\n• Alerting & on-call',
    '#27AE60')

# Developer Tools (bottom)
box(ax, 0.3, 2.8, 15.4, 1.8, 'DEVELOPER EXPERIENCE LAYER',
    '• Python SDK / CLI  • Notebook integration  • CI/CD for ML  • Documentation  • Cost visibility',
    '#34495E')

# User types
box(ax, 0.3, 0.8, 7, 1.5, 'DATA SCIENTISTS',
    'Use: Feature eng, training, evaluation, experimentation',
    '#7F8C8D')
box(ax, 8.3, 0.8, 7.4, 1.5, 'ML ENGINEERS / MLOPS',
    'Own: Platform infra, serving, monitoring, reliability',
    '#7F8C8D')

# Arrows
for x in [2.55, 7.45, 12.9]:
    ax.annotate('', xy=(x, 8.8), xytext=(x, 8.6),
                arrowprops=dict(arrowstyle='->', color='#aaa', lw=1.5))

plt.tight_layout()
plt.savefig('ml_platform_components.png', dpi=120, bbox_inches='tight')
plt.show()

## 2. Real-World ML Platforms: Case Studies

Studying how major tech companies designed their ML platforms reveals common patterns and tradeoffs. Most platforms converged on similar architectures independently, validating the core design choices.

In [ ]:
platforms = [
    {
        'company': 'Uber',
        'platform': 'Michelangelo',
        'year': 2017,
        'key_innovations': [
            'First published end-to-end ML platform architecture',
            'Centralized feature store (Palette) — seminal for the concept',
            'Unified offline/online feature serving',
            'Standardized training + serving workflow across all teams',
        ],
        'tech_stack': 'Spark, Cassandra, HDFS, Flink, Kafka, TensorFlow',
        'scale': 'Thousands of models, hundreds of use cases',
        'key_lesson': 'Centralized feature store eliminated most training-serving skew',
    },
    {
        'company': 'Meta',
        'platform': 'FBLearner Flow',
        'year': 2016,
        'key_innovations': [
            'ML workflow as composable Python DAGs',
            'One-click model training from feature registration to serving',
            'Automatic A/B test launch from platform',
            'Built-in ONNX model portability',
        ],
        'tech_stack': 'Python, Daimon (internal), internal feature stores',
        'scale': '10s of thousands of trained models per month',
        'key_lesson': 'Workflow reusability dramatically reduced time-to-production',
    },
    {
        'company': 'Airbnb',
        'platform': 'Bighead',
        'year': 2018,
        'key_innovations': [
            'Zipline feature pipeline: time-series aggregations abstracted',
            'Unified feature definition in Python, deployed to Spark/online',
            'AutoML capabilities for baseline model creation',
            'Strong lineage: feature → model → prediction tracking',
        ],
        'tech_stack': 'Python, Spark, Airflow, Zipline, Python serving',
        'scale': '150+ models in production',
        'key_lesson': 'Abstracting time-series feature computation saved months per team',
    },
    {
        'company': 'LinkedIn',
        'platform': 'Pro-ML / Feathr',
        'year': 2019,
        'key_innovations': [
            'Feathr (open-sourced 2022): unified feature definition language',
            'Support for both Spark (offline) and Redis (online) from one spec',
            'Tight integration with Apache Spark and Azure ML',
            'Feature sharing across teams via centralized registry',
        ],
        'tech_stack': 'Spark, Redis, Azure ML, Kafka, open-sourced Feathr',
        'scale': 'Hundreds of models, 10B+ feature retrievals/day',
        'key_lesson': 'Open-sourcing platform components accelerated adoption and contribution',
    },
]

for p in platforms:
    print(f"\n{'='*70}")
    print(f"  {p['company']} — {p['platform']} ({p['year']})")
    print(f"{'='*70}")
    print(f"  Tech: {p['tech_stack']}")
    print(f"  Scale: {p['scale']}")
    print(f"  Key innovations:")
    for inn in p['key_innovations']:
        print(f"    • {inn}")
    print(f"  Lesson: {p['key_lesson']}")

## 3. Platform Build vs Buy Decision Framework

The build vs buy decision for ML platform components is one of the most consequential architectural choices. Most companies start with open-source or managed tools and graduate to custom platforms as they scale.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Build vs Buy analysis by platform component and company scale
components = [
    'Feature Store',
    'Training Orchestration',
    'Experiment Tracking',
    'Model Registry',
    'Serving Platform',
    'Monitoring',
    'Data Pipelines',
]

# Score 1=strongly buy, 5=strongly build, by company scale
startup_scores = [2, 2, 1, 1, 2, 1, 2]    # Startups: buy everything
mid_scores =     [3, 3, 2, 2, 3, 2, 3]    # Mid-stage: selective buy
large_scores =   [4, 4, 3, 3, 4, 4, 4]    # Large: custom where differentiating
hyper_scores =   [5, 5, 4, 4, 5, 5, 5]    # Hyper-scale: mostly build

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

ax = axes[0]
x = np.arange(len(components))
width = 0.2

bars1 = ax.bar(x - 1.5*width, startup_scores, width, label='Startup (<10 ML engineers)', color='#27AE60', alpha=0.8)
bars2 = ax.bar(x - 0.5*width, mid_scores, width, label='Mid-stage (10-50 ML engineers)', color='#F39C12', alpha=0.8)
bars3 = ax.bar(x + 0.5*width, large_scores, width, label='Large (50-200 ML engineers)', color='#E67E22', alpha=0.8)
bars4 = ax.bar(x + 1.5*width, hyper_scores, width, label='Hyper-scale (200+ ML engineers)', color='#E74C3C', alpha=0.8)

ax.axhline(y=3, color='gray', linestyle='--', alpha=0.7, label='Build/Buy threshold')
ax.set_xlabel('Platform Component', fontsize=10)
ax.set_ylabel('Build vs Buy Score (1=Buy, 5=Build)', fontsize=10)
ax.set_title('Build vs Buy by Component and Scale', fontsize=11, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(components, rotation=20, ha='right', fontsize=8)
ax.legend(fontsize=8, loc='upper left')
ax.set_ylim(0, 6)
ax.grid(True, axis='y', alpha=0.3)

# Option comparison table
ax2 = axes[1]
ax2.set_xlim(0, 10); ax2.set_ylim(0, len(components) * 1.5 + 0.5); ax2.axis('off')
ax2.set_title('Recommended Options by Component', fontsize=11, fontweight='bold')

options = [
    ('Feature Store', 'Feast (OSS)', 'Tecton/Hopsworks', 'Custom (Airbnb Zipline)'),
    ('Training Orch.', 'Metaflow/Prefect', 'Kubeflow Pipelines', 'Custom DAG runner'),
    ('Experiment Track', 'MLflow (OSS)', 'W&B / Neptune', 'Custom (FBLearner)'),
    ('Model Registry', 'MLflow Registry', 'SageMaker/Vertex', 'Custom artifact store'),
    ('Serving', 'BentoML/TorchServe', 'SageMaker/Vertex', 'Custom (Michelangelo serve)'),
    ('Monitoring', 'Evidently/Arize', 'Fiddler/WhyLabs', 'Custom metrics pipeline'),
    ('Data Pipelines', 'Airflow/Prefect', 'Dagster/Astronomer', 'Custom orchestrator'),
]

headers = ['Component', 'Startup/Open Source', 'Mid-stage/Managed', 'Large/Custom']
col_x = [0.1, 2.8, 5.4, 7.9]
for j, h in enumerate(headers):
    ax2.text(col_x[j], ax2.get_ylim()[1] - 0.4, h, fontsize=9, fontweight='bold', color='#333')

from matplotlib.patches import FancyBboxPatch
row_colors = ['#EBF5FB', '#E8F8F5', '#FDF2E9', '#F9EBEA', '#F5EEF8', '#EAFAF1', '#FEF9E7']
for i, (comp, opt1, opt2, opt3) in enumerate(options):
    y = ax2.get_ylim()[1] - 1.5 - i * 1.35
    ax2.add_patch(FancyBboxPatch((0.05, y - 0.5), 9.9, 1.2, boxstyle="round,pad=0.05",
                                  facecolor=row_colors[i], edgecolor='#ddd', linewidth=1))
    ax2.text(col_x[0], y + 0.15, comp, fontsize=8.5, fontweight='bold', color='#333')
    ax2.text(col_x[1], y + 0.15, opt1, fontsize=8, color='#27AE60')
    ax2.text(col_x[2], y + 0.15, opt2, fontsize=8, color='#E67E22')
    ax2.text(col_x[3], y + 0.15, opt3, fontsize=8, color='#E74C3C')

plt.tight_layout()
plt.savefig('build_vs_buy.png', dpi=120, bbox_inches='tight')
plt.show()

print("\nKey decision factors:")
factors = [
    ("Team size", "< 10 ML engineers → buy everything; > 50 → selective build"),
    ("Differentiation", "Build only what gives competitive advantage"),
    ("Data sensitivity", "Regulated industries may need on-prem → lean toward build"),
    ("Velocity", "Startups need speed → buy all; scale → build bottlenecks"),
    ("Cost at scale", "Managed services can be 5-10× more expensive at hyper-scale"),
]
for factor, guidance in factors:
    print(f"  {factor}: {guidance}")

## 4. Platform Design Principles

The most successful ML platforms share common design principles—learned through painful experience at companies like Google (which published these in its ML engineering practices research).

In [ ]:
principles = {
    "Make the Right Way the Easy Way": {
        "description": "Best practices should be defaults, not opt-ins",
        "examples": [
            "Feature validation is automatic — teams can't skip it",
            "Experiment tracking is always on — not optional",
            "Canary deployment is the default path to production",
            "Data lineage is captured automatically, not manually",
        ],
        "anti_pattern": "Providing best-practice templates no one uses"
    },
    "Separation of Concerns": {
        "description": "Data scientists define WHAT; platform handles HOW",
        "examples": [
            "DS defines feature logic in Python; platform handles Spark vs Flink vs online",
            "DS defines training code; platform handles distributed execution",
            "DS defines model; platform handles serialization/versioning",
        ],
        "anti_pattern": "Requiring DS to write Kubernetes YAML or Spark jobs"
    },
    "Reproducibility by Default": {
        "description": "Every run must be reproducible without extra effort",
        "examples": [
            "Data version is always captured with the run",
            "Code version (git SHA) is always captured",
            "Hyperparameters logged even when not explicitly called",
            "Random seeds are logged and can be replayed",
        ],
        "anti_pattern": "Asking DS to remember to log data versions"
    },
    "Gradual Escape Hatches": {
        "description": "Standardized paths for 80%, escape hatches for the 20%",
        "examples": [
            "Standard training jobs; custom Docker containers for edge cases",
            "Standard REST serving; custom compiled models allowed",
            "Standard feature pipelines; raw SQL escape hatch for complex transformations",
        ],
        "anti_pattern": "All-or-nothing: use the platform exactly as designed or DIY everything"
    },
    "Cost Visibility": {
        "description": "Teams must see the cost of their models and features",
        "examples": [
            "Show compute cost per training run",
            "Show serving cost per model per day",
            "Show feature computation cost per pipeline",
            "Budget alerts when cost exceeds threshold",
        ],
        "anti_pattern": "Central infra absorbs all costs with no team accountability"
    },
}

for principle, details in principles.items():
    print(f"\n{'='*65}")
    print(f"  Principle: {principle}")
    print(f"  → {details['description']}")
    print(f"{'='*65}")
    print("  Examples:")
    for ex in details['examples']:
        print(f"    ✓ {ex}")
    print(f"  Anti-pattern: ✗ {details['anti_pattern']}")

## 5. MLOps Maturity Model

Google's MLOps whitepaper defines maturity levels that describe how organizations progress from manual ML workflows to fully automated, self-healing ML systems.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(15, 10))
ax.set_xlim(0, 15); ax.set_ylim(0, 10); ax.axis('off')
ax.set_title('MLOps Maturity Model (Google Framework)', fontsize=14, fontweight='bold')

levels = [
    {
        'level': 'Level 0',
        'name': 'Manual Process',
        'color': '#E74C3C',
        'y': 7.2,
        'traits': [
            'Manual, script-driven training',
            'No CI/CD for ML code',
            'Model deployed manually',
            'No monitoring',
            'Disconnected from data pipeline',
        ],
        'companies': 'Most startups, early ML teams',
    },
    {
        'level': 'Level 1',
        'name': 'ML Pipeline Automation',
        'color': '#E67E22',
        'y': 4.5,
        'traits': [
            'Automated training pipeline (DAG)',
            'Feature store in place',
            'Model registry',
            'Triggered retraining (drift or schedule)',
            'Basic monitoring in place',
        ],
        'companies': 'Mid-stage ML teams, most SaaS companies',
    },
    {
        'level': 'Level 2',
        'name': 'CI/CD Pipeline Automation',
        'color': '#27AE60',
        'y': 1.8,
        'traits': [
            'CI/CD pipeline for ML code AND data AND model',
            'Automated deployment with canary + rollback',
            'Full lineage: data → features → model → predictions',
            'Continuous training + continuous evaluation',
            'Self-healing: drift → retrain → deploy automatically',
        ],
        'companies': 'Google, Meta, Uber, Airbnb at their maturity peaks',
    },
]

for lvl in levels:
    ax.add_patch(FancyBboxPatch((0.3, lvl['y']), 14.4, 2.5,
                                boxstyle="round,pad=0.15", facecolor=lvl['color'],
                                edgecolor='white', linewidth=2, alpha=0.12))
    ax.add_patch(FancyBboxPatch((0.3, lvl['y']), 2, 2.5,
                                boxstyle="round,pad=0.1", facecolor=lvl['color'],
                                edgecolor='white', linewidth=2, alpha=0.9))
    ax.text(1.3, lvl['y'] + 1.7, lvl['level'], ha='center', va='center',
            fontsize=10, fontweight='bold', color='white')
    ax.text(1.3, lvl['y'] + 1.0, lvl['name'], ha='center', va='center',
            fontsize=8, color='white', multialignment='center')

    for i, trait in enumerate(lvl['traits']):
        ax.text(2.6, lvl['y'] + 2.15 - i * 0.42, f'• {trait}',
                fontsize=8.5, color='#333')

    ax.text(10.5, lvl['y'] + 0.4, f'Companies: {lvl["companies"]}',
            fontsize=8, color='#666', style='italic')

# Maturity arrows
ax.annotate('', xy=(1.3, 4.5), xytext=(1.3, 7.2),
            arrowprops=dict(arrowstyle='->', color=levels[0]['color'], lw=2.5))
ax.annotate('', xy=(1.3, 1.8), xytext=(1.3, 4.5),
            arrowprops=dict(arrowstyle='->', color=levels[1]['color'], lw=2.5))

ax.text(7.5, 0.5, 'Most organizations target Level 1; Level 2 requires sustained platform investment',
        ha='center', fontsize=9, color='#555', style='italic')

plt.tight_layout()
plt.savefig('mlops_maturity.png', dpi=120, bbox_inches='tight')
plt.show()

print("\nKey transitions:")
transitions = [
    ("Level 0 → 1", "Build training pipeline, feature store, model registry, basic monitoring"),
    ("Level 1 → 2", "Add CI/CD for ML, automated deployment, continuous training + evaluation"),
]
for transition, what_needed in transitions:
    print(f"  {transition}: {what_needed}")

print("\nCommon Level 0 organization pain points:")
pains = [
    "'Works on my machine' — model can't be reproduced",
    "Model deployed 3 months after training — already stale",
    "Feature logic duplicated in training notebook and Java service",
    "No one knows what data version was used for the model in prod",
    "Performance regression discovered by users, not monitoring",
]
for p in pains:
    print(f"  ✗ {p}")

## Summary

| Component | Purpose | Key Tools | When to Build Custom |
|-----------|---------|-----------|---------------------|
| Feature Store | Eliminate training-serving skew | Feast, Tecton, Hopsworks | > 100 features, streaming requirements |
| Training Platform | Scalable, reproducible training | Kubeflow, Metaflow, Vertex AI | > 50 DS, unique infra requirements |
| Experiment Tracking | Reproducibility, comparison | MLflow, W&B | Rarely — MLflow covers most needs |
| Model Registry | Version management, staging | MLflow, SageMaker | When compliance requires custom audit |
| Serving Platform | Low-latency, scalable inference | TorchServe, Triton, BentoML | > 100 models, strict SLAs |
| Monitoring | Drift detection, alerting | Evidently, Arize, Fiddler | Custom business metrics at scale |

**Key Principles**
- Make best practices defaults, not opt-ins
- Separate DS concerns (what) from platform concerns (how)
- Reproducibility must be automatic, not voluntary
- Provide escape hatches—no platform handles 100% of use cases
- Show teams their compute and serving costs
- Most organizations should buy before they build